# 📊 BÀI TẬP GIỮA KỲ: TIỀN XỬ LÝ VÀ KHÁM PHÁ DỮ LIỆU (EDA)
## ĐỀ TÀI: PHÂN TÍCH HÀNH VI & NGUY CƠ RỜI BỎ CỦA KHÁCH HÀNG THẺ TÍN DỤNG
**Học phần:** Phân tích dữ liệu trong kinh doanh (Business Analytics)  
**Đơn vị:** Học viện Ngân hàng (Banking Academy)  
**Tập dữ liệu:**  (10,127 quan sát, 23 thuộc tính)  

---
### 🎯 Mục tiêu nghiên cứu:
1. **Làm sạch & Tiền xử lý dữ liệu:** Khắc phục giá trị khuyết thiếu, loại bỏ thuộc tính nhiễu, mã hóa biến và kỹ thuật tạo biến mới (*Feature Engineering*).
2. **Khám phá dữ liệu (EDA):** Thống kê mô tả định tính, định lượng, phân tích đơn biến, hai biến và đa biến.
3. **Nhận diện nguyên nhân rời bỏ:** Tìm ra các chỉ báo hành vi then chốt (*Key Churn Indicators*) dẫn đến việc khách hàng đóng thẻ.
4. **Hàm ý quản trị:** Xây dựng chân dung khách hàng rủi ro (*Churn Persona*) và đề xuất chiến lược giữ chân (*Customer Retention Strategy*) cho Ngân hàng.


## 🛠️ 1. Khởi tạo môi trường & Đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị và đồ họa
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Đọc tập dữ liệu
df_raw = pd.read_csv('BankChurners.csv')
print(f'✅ Tải dữ liệu thành công! Kích thước: {df_raw.shape[0]:,} dòng, {df_raw.shape[1]} cột.')
df_raw.head()


## 📌 2. Tổng quan cấu trúc dữ liệu & Từ điển biến

In [ ]:
# Bảng tổng hợp cấu trúc dữ liệu
summary_df = pd.DataFrame({
    'Cột': df_raw.columns,
    'Kiểu dữ liệu': df_raw.dtypes.values,
    'Số lượng Non-Null': df_raw.notnull().sum().values,
    'Số lượng Missing': df_raw.isnull().sum().values,
    'Tỷ lệ Missing (%)': (df_raw.isnull().sum().values / len(df_raw)) * 100,
    'Số giá trị Unique': df_raw.nunique().values
})
summary_df


## 🧹 3. Tiền xử lý & Làm sạch dữ liệu (Data Preprocessing)
1. **Loại bỏ cột nhiễu:** 2 cột Naive Bayes Classifier ở cuối bảng được thêm sẵn từ Kaggle, không thuộc dữ liệu gốc.
2. **Xử lý giá trị khuyết thiếu (Missing Values):** Điền giá trị trung vị (Median) cho các thuộc tính định lượng.
3. **Kỹ thuật tạo biến mới (Feature Engineering):**
   - : Phân nhóm độ tuổi khách hàng.
   - : Giá trị chi tiêu trung bình trên mỗi giao dịch ().
   - : Mã hóa nhị phân (1: Rời bỏ, 0: Ở lại).


In [ ]:
# 1. Loại bỏ các cột thừa Naive Bayes
cols_to_drop = [c for c in df_raw.columns if c.startswith('Naive_Bayes_Classifier')]
df = df_raw.drop(columns=cols_to_drop).copy()

# 2. Xử lý Missing values bằng Median
if df['Months_Inactive_12_mon'].isnull().sum() > 0:
    df['Months_Inactive_12_mon'].fillna(df['Months_Inactive_12_mon'].median(), inplace=True)
if df['Contacts_Count_12_mon'].isnull().sum() > 0:
    df['Contacts_Count_12_mon'].fillna(df['Contacts_Count_12_mon'].median(), inplace=True)
if df['Total_Revolving_Bal'].isnull().sum() > 0:
    df['Total_Revolving_Bal'].fillna(df['Total_Revolving_Bal'].median(), inplace=True)

# 3. Feature Engineering
df['Age_Group'] = pd.cut(
    df['Customer_Age'], 
    bins=[20, 35, 45, 55, 65, 80], 
    labels=['<35 (Trẻ)', '35-44 (Trung niên trẻ)', '45-54 (Trung niên)', '55-64 (Lớn tuổi)', '65+ (Cao tuổi)']
)
df['Avg_Trans_Value'] = df['Total_Trans_Amt'] / df['Total_Trans_Ct']
df['Churn_Binary'] = (df['Attrition_Flag'] == 'Attrited Customer').astype(int)

print(f'✅ Kích thước sau khi làm sạch: {df.shape[0]:,} dòng, {df.shape[1]} cột.')
print(f'Tổng số giá trị khuyết thiếu còn lại: {df.isnull().sum().sum()}')


## 📈 4. Khám phá Dữ liệu (EDA) - Phân tích Biến Mục tiêu (Attrition Flag)

In [ ]:
churn_counts = df['Attrition_Flag'].value_counts()
churn_pct = df['Attrition_Flag'].value_counts(normalize=True) * 100

print('--- PHÂN PHỐI TRẠNG THÁI KHÁCH HÀNG ---')
for k in churn_counts.index:
    print(f'{k}: {churn_counts[k]:,} khách hàng ({churn_pct[k]:.2f}%)')

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#1E3A8A', '#EF4444']
ax.pie(churn_counts, labels=['Existing Customer', 'Attrited Customer'], autopct='%1.2f%%', startangle=140, colors=colors,
       wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2), textprops=dict(fontweight='bold', fontsize=11))
ax.set_title('Tỷ lệ Khách hàng Rời bỏ Thẻ (Attrition Rate)', fontsize=13, fontweight='bold')
plt.show()


## 👥 5. Phân tích Nhân khẩu học & Tỷ lệ Rời bỏ (Demographics Analysis)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
palette = {'Existing Customer': '#1E3A8A', 'Attrited Customer': '#EF4444'}

# 1. Phân bố tuổi KDE
sns.kdeplot(data=df, x='Customer_Age', hue='Attrition_Flag', common_norm=False, fill=True, palette=palette, alpha=0.4, ax=axes[0, 0], linewidth=2)
axes[0, 0].set_title('Phân bố Độ tuổi (Customer Age)', fontweight='bold')

# 2. Giới tính
gender_cross = pd.crosstab(df['Gender'], df['Attrition_Flag'], normalize='index') * 100
gender_cross.plot(kind='bar', stacked=True, color=['#EF4444', '#1E3A8A'], ax=axes[0, 1])
axes[0, 1].set_title('Tỷ lệ Rời bỏ theo Giới tính (Gender)', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. Thu nhập
income_cross = pd.crosstab(df['Income_Category'], df['Attrition_Flag'], normalize='index') * 100
income_cross.plot(kind='bar', stacked=True, color=['#EF4444', '#1E3A8A'], ax=axes[1, 0])
axes[1, 0].set_title('Tỷ lệ Rời bỏ theo Mức thu nhập (Income Category)', fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=30)

# 4. Trình độ học vấn
edu_cross = pd.crosstab(df['Education_Level'], df['Attrition_Flag'], normalize='index') * 100
edu_cross.plot(kind='bar', stacked=True, color=['#EF4444', '#1E3A8A'], ax=axes[1, 1])
axes[1, 1].set_title('Tỷ lệ Rời bỏ theo Trình độ học vấn (Education Level)', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 💳 6. Phân tích Hành vi Giao dịch & Sử dụng Thẻ (Transaction Behavior)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='Total_Trans_Ct', y='Total_Trans_Amt', hue='Attrition_Flag', palette=palette, alpha=0.6, s=35, ax=ax)
ax.set_title('Tổng số lần giao dịch (Trans Ct) vs Tổng giá trị giao dịch (Trans Amt)', fontweight='bold', fontsize=13)
ax.set_xlabel('Tổng số giao dịch (Total_Trans_Ct)')
ax.set_ylabel('Tổng giá trị giao dịch ($) (Total_Trans_Amt)')
ax.axvline(x=55, color='gray', linestyle='--', alpha=0.7)
ax.axhline(y=3000, color='gray', linestyle='--', alpha=0.7)
plt.show()


## 📉 7. Phân tích Tương tác & Dư nợ Tín dụng (Engagement & Balance)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Dư nợ quay vòng
sns.boxplot(data=df, x='Attrition_Flag', y='Total_Revolving_Bal', palette=palette, ax=axes[0, 0])
axes[0, 0].set_title('Dư nợ tín dụng quay vòng ($)', fontweight='bold')

# Tỷ lệ sử dụng hạn mức
sns.boxplot(data=df, x='Attrition_Flag', y='Avg_Utilization_Ratio', palette=palette, ax=axes[0, 1])
axes[0, 1].set_title('Tỷ lệ sử dụng hạn mức (Avg Utilization Ratio)', fontweight='bold')

# Số lần liên hệ
contacts_cross = pd.crosstab(df['Contacts_Count_12_mon'], df['Attrition_Flag'], normalize='index') * 100
contacts_cross.plot(kind='bar', stacked=True, color=['#EF4444', '#1E3A8A'], ax=axes[1, 0])
axes[1, 0].set_title('Tỷ lệ Rời bỏ theo Số lần liên hệ CSKH', fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=0)

# Số tháng không hoạt động
inactive_cross = pd.crosstab(df['Months_Inactive_12_mon'], df['Attrition_Flag'], normalize='index') * 100
inactive_cross.plot(kind='bar', stacked=True, color=['#EF4444', '#1E3A8A'], ax=axes[1, 1])
axes[1, 1].set_title('Tỷ lệ Rời bỏ theo Số tháng không hoạt động', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


## 🔍 8. Ma trận tương quan Pearson & Kiểm định Thống kê (Hypothesis Testing)

In [ ]:
num_cols = [
    'Customer_Age', 'Dependent_count', 'Months_on_book', 'Total_Relationship_Count',
    'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
    'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct',
    'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Churn_Binary'
]

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Ma trận tương quan Pearson giữa các biến số tài chính & Churn', fontweight='bold', fontsize=13)
plt.show()


In [ ]:
# Bảng kiểm định thống kê T-test so sánh 2 nhóm
stats_list = []
for col in [c for c in num_cols if c != 'Churn_Binary']:
    g1 = df[df['Attrition_Flag'] == 'Existing Customer'][col]
    g2 = df[df['Attrition_Flag'] == 'Attrited Customer'][col]
    t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=False)
    stats_list.append({
        'Thuộc tính': col,
        'Existing Mean': g1.mean(),
        'Attrited Mean': g2.mean(),
        'Chênh lệch (%)': ((g2.mean() - g1.mean()) / g1.mean()) * 100,
        'p-value': p_val,
        'Mức ý nghĩa': '*** (p < 0.001)' if p_val < 0.001 else ('* (p < 0.05)' if p_val < 0.05 else 'ns (Không có ý nghĩa)')
    })

pd.DataFrame(stats_list)


## 💡 9. Tổng kết & Chân dung Khách hàng Rời bỏ (Customer Churn Persona)

### 📌 Chân dung Khách hàng Nguy cơ Cao (High-Risk Profile):
1. **Hành vi quẹt thẻ suy giảm mạnh:** Số lượng giao dịch $\le 45$ lần/năm, sụt giảm giao dịch quý 4 so với quý 1 $> 45\%$.
2. **Dư nợ quay vòng tiệm cận zsh:** Không còn giữ số dư vay nợ trên thẻ, tỷ lệ sử dụng hạn mức $< 10\%$.
3. **Tín hiệu bức xúc / khiếu nại:** Số lần gọi lên tổng đài $\ge 3$ lần/năm (tỷ lệ rời bỏ vọt lên $> 30\%$).
4. **Ngủ đông kéo dài:** Không phát sinh giao dịch $\ge 3$ tháng liên tiếp.
5. **Gắn kết nông:** Sử dụng $\le 3$ sản phẩm/dịch vụ của ngân hàng.

### 🎯 Chiến lược Giữ chân (Retention Action Plan):
- **Cảnh báo sớm (Early Warning Triggers):** Kích hoạt chăm sóc đặc biệt khi phát hiện giao dịch giảm liên tiếp 2 tháng hoặc khách gọi CSKH lần thứ 3.
- **Ưu đãi kích hoạt lại (Re-engagement Campaigns):** Miễn phí thường niên, hoàn tiền 5% cho các chi tiêu thiết yếu (siêu thị, xăng dầu).
- **Gia tăng gắn kết (Cross-selling):** Gói tích hợp tài khoản thanh toán + bảo hiểm + thẻ tín dụng để tăng  $\ge 4$.
